# Development of State Estimators

In [ ]:
import casadi as cas
import numpy as np
import scipy


## Linear Kalman filter equations

In [ ]:
from cas_models.discrete_time.kalman import (
    kalman_predict_f,
    kalman_update_f,
    kf_update,
    linear_model_output,
    linear_model_state_transition,
    make_steady_state_KFP_with_fixedABCDQR,
    make_steady_state_KFF_with_fixedABCDQR,
    KFP,
    KFF
)

## Unit tests

In [ ]:
# Test system used in the tests below: 2 states, 1 input, 2 outputs
A_test = np.array([[0.7, 1.0], [0.0, 1.0]])
B_test = np.array([[1.0], [0.0]])
C_test = np.array([[0.3, 0.0], [0.0, 1.0]])
D_test = np.zeros((2, 1))
Q_test = np.diag([0.01, 0.01])
R_test = np.diag([0.1, 0.1])


def np_kalman_update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk):
    """NumPy version of the original JAX code (1-D vectors),
    used as a reference to test the CasADi functions.
    """
    yk_pred = C @ xkp1_est + D @ uk
    Sk_pred = C @ Pkp1 @ C.T + R
    Kf = np.linalg.lstsq(Sk_pred.T, (Pkp1 @ C.T).T, rcond=-1)[0].T
    xk_est = xkp1_est + Kf @ (yk - yk_pred)
    z = np.eye(A.shape[0]) - Kf @ C
    Pk = z @ Pkp1 @ z.T + Kf @ R @ Kf.T
    Pk = (Pk + Pk.T) / 2
    xkp1_est = A @ xk_est + B @ uk
    Pkp1 = A @ Pk @ A.T + Q
    return xk_est, Pk, xkp1_est, Pkp1, yk_pred, Kf

In [ ]:
def test_linear_model_functions():
    A, B, C, D = map(cas.DM, (A_test, B_test, C_test, D_test))
    xk = cas.DM([1.0, 2.0])
    uk = cas.DM([0.5])

    xkp1 = linear_model_state_transition(A, B, xk, uk)
    assert isinstance(xkp1, cas.DM)
    assert xkp1.shape == (2, 1)
    np.testing.assert_allclose(xkp1, [[0.7 + 2.0 + 0.5], [2.0]])

    yk = linear_model_output(C, D, xk, uk)
    assert yk.shape == (2, 1)
    np.testing.assert_allclose(yk, [[0.3], [2.0]])

    # Non-zero D
    D = cas.DM([[1.0], [-1.0]])
    yk = linear_model_output(C, D, xk, uk)
    np.testing.assert_allclose(yk, [[0.3 + 0.5], [2.0 - 0.5]])

    # No inputs (nu = 0)
    xkp1 = linear_model_state_transition(
        A, cas.DM.zeros(2, 0), xk, cas.DM.zeros(0)
    )
    np.testing.assert_allclose(xkp1, A_test @ [[1.0], [2.0]])

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        xs, us = T.sym("x", 2), T.sym("u", 1)
        f = cas.Function(
            "f", [xs, us], [linear_model_state_transition(A, B, xs, us)]
        )
        np.testing.assert_allclose(f(xk, uk), [[3.2], [2.0]])


test_linear_model_functions()
print("test_linear_model_functions passed")

In [ ]:
def test_kalman_update_f():
    # Scalar system with known solution:
    # S = 1 * 1 * 1 + 1 = 2, Kf = 1 / 2,
    # x(k|k) = 0 + 0.5 * (4 - 0) = 2,
    # P(k|k) = 0.5^2 * 1 + 0.5^2 * 1 = 0.5
    one = cas.DM(1.0)
    xk_est, Pk, Kf, Sk = kalman_update_f(
        one, one, cas.DM(0.0), one, cas.DM(0.0), cas.DM(4.0)
    )
    for value in (xk_est, Pk, Kf, Sk):
        assert isinstance(value, cas.DM)
        assert value.shape == (1, 1)
    np.testing.assert_allclose(Sk, 2.0)
    np.testing.assert_allclose(Kf, 0.5)
    np.testing.assert_allclose(xk_est, 2.0)
    np.testing.assert_allclose(Pk, 0.5)

    # 2-state system compared to the standard form of the equations
    C, R = cas.DM(C_test), cas.DM(R_test)
    xk_pred = cas.DM([1.0, -1.0])
    Pk_pred = cas.DM([[2.0, 0.5], [0.5, 1.0]])
    yk_pred = C @ xk_pred
    yk = cas.DM([0.5, 0.0])
    xk_est, Pk, Kf, Sk = kalman_update_f(C, R, xk_pred, Pk_pred, yk_pred, yk)
    P, Cn = np.array(Pk_pred), C_test
    S = Cn @ P @ Cn.T + R_test
    K = P @ Cn.T @ np.linalg.inv(S)
    assert Kf.shape == (2, 2)
    np.testing.assert_allclose(Sk, S)
    np.testing.assert_allclose(Kf, K)
    np.testing.assert_allclose(
        xk_est, np.array(xk_pred) + K @ np.array(yk - yk_pred)
    )
    np.testing.assert_allclose(Pk, P - K @ S @ K.T)
    np.testing.assert_allclose(Pk, Pk.T)

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        x, P = T.sym("x", 2), T.sym("P", 2, 2)
        y = T.sym("y", 2)
        f = cas.Function("f", [x, P, y], kalman_update_f(C, R, x, P, C @ x, y))
        for a, b in zip(f(xk_pred, Pk_pred, yk), (xk_est, Pk, Kf, Sk)):
            np.testing.assert_allclose(a, b)


test_kalman_update_f()
print("test_kalman_update_f passed")

In [ ]:
def test_kalman_predict_f():
    A, B, Q = map(cas.DM, (A_test, B_test, Q_test))
    xk_est = cas.DM([1.0, 2.0])
    Pk = cas.DM([[2.0, 0.5], [0.5, 1.0]])
    uk = cas.DM([0.5])

    xkp1_est, Pkp1 = kalman_predict_f(A, B, Q, xk_est, Pk, uk)
    assert xkp1_est.shape == (2, 1)
    assert Pkp1.shape == (2, 2)
    np.testing.assert_allclose(xkp1_est, [[3.2], [2.0]])
    np.testing.assert_allclose(Pkp1, A_test @ np.array(Pk) @ A_test.T + Q_test)

    # Symbolic arguments
    for T in (cas.SX, cas.MX):
        x, P, u = T.sym("x", 2), T.sym("P", 2, 2), T.sym("u", 1)
        f = cas.Function("f", [x, P, u], kalman_predict_f(A, B, Q, x, P, u))
        for a, b in zip(f(xk_est, Pk, uk), (xkp1_est, Pkp1)):
            np.testing.assert_allclose(a, b)


test_kalman_predict_f()
print("test_kalman_predict_f passed")

In [ ]:
def test_update():
    A, B, C, D, Q, R = map(
        cas.DM, (A_test, B_test, C_test, D_test, Q_test, R_test)
    )

    # Compare to NumPy reference over 50 time steps with random data
    rng = np.random.default_rng(0)
    xk_pred, Pk_pred = cas.DM.zeros(2), cas.DM.eye(2)
    xk_pred_np, Pk_pred_np = np.zeros(2), np.eye(2)
    for k in range(50):
        yk, uk = rng.normal(size=2), rng.normal(size=1)
        results = kf_update(A, B, C, D, xk_pred, Pk_pred, Q, R, yk, uk)
        results_np = np_kalman_update(
            A_test,
            B_test,
            C_test,
            D_test,
            xk_pred_np,
            Pk_pred_np,
            Q_test,
            R_test,
            yk,
            uk,
        )
        # Previously: xk_est, Pk, yk_est, Sk, xkp1_est, Pkp1, yk_pred, Kf
        # Now: xk_est, Pk, xkp1_est, Pkp1, yk_pred, Kf
        for a, b in zip(results, results_np):
            assert isinstance(a, cas.DM)
            np.testing.assert_allclose(
                np.array(a).reshape(b.shape), b, atol=1e-12
            )
        xkp1_est, Pkp1 = results[2], results[3]
        xkp1_est_np, Pkp1_np = results_np[2], results_np[3]

        # Update variables for next iteration
        xk_pred, Pk_pred = xkp1_est, Pkp1
        xk_pred_np, Pk_pred_np =  xkp1_est_np, Pkp1_np

    # After many steps the gain should converge to the steady-state
    # Kalman filter gain from the discrete algebraic Riccati equation
    P_ss = scipy.linalg.solve_discrete_are(A_test.T, C_test.T, Q_test, R_test)
    Kf_ss = P_ss @ C_test.T @ np.linalg.inv(C_test @ P_ss @ C_test.T + R_test)
    np.testing.assert_allclose(Pkp1, P_ss, atol=1e-8)
    np.testing.assert_allclose(results[5], Kf_ss, atol=1e-8)

    # Lists and floats are accepted as yk and uk
    results = kf_update(A, B, C, D, xkp1_est, Pkp1, Q, R, [1.0, 2.0], 0.5)
    assert results[0].shape == (2, 1)

    # No inputs (nu = 0), uk omitted
    results = kf_update(
        A,
        cas.DM.zeros(2, 0),
        C,
        cas.DM.zeros(2, 0),
        cas.DM.zeros(2),
        cas.DM.eye(2),
        Q,
        R,
        [1.0, 2.0],
    )
    results_np = np_kalman_update(
        A_test,
        np.zeros((2, 0)),
        C_test,
        np.zeros((2, 0)),
        np.zeros(2),
        np.eye(2),
        Q_test,
        R_test,
        np.array([1.0, 2.0]),
        np.zeros(0),
    )
    for a, b in zip(results, results_np):
        np.testing.assert_allclose(np.array(a).reshape(b.shape), b)

    # Wrong-sized measurement raises an error
    try:
        kf_update(A, B, C, D, xkp1_est, Pkp1, Q, R, [1.0], 0.5)
    except AssertionError as e:
        assert str(e) == "yk.shape"
    else:
        raise AssertionError("expected AssertionError")

    # Symbolic arguments: build a cas.Function of one time step
    for T in (cas.SX, cas.MX):
        x, P = T.sym("x", 2), T.sym("P", 2, 2)
        y, u = T.sym("y", 2), T.sym("u", 1)
        f = cas.Function(
            "kf_update",
            [x, P, y, u],
            list(kf_update(A, B, C, D, x, P, Q, R, y, u)),
        )
        expected = kf_update(A, B, C, D, xkp1_est, Pkp1, Q, R, yk, uk)
        for a, b in zip(f(xkp1_est, Pkp1, yk, uk), expected):
            np.testing.assert_allclose(a, b)


test_update()
print("test_update passed")

In [ ]:
A, B, C, D, Q, R = map(
    cas.DM, (A_test, B_test, C_test, D_test, Q_test, R_test)
)

kfpss = make_steady_state_KFP_with_fixedABCDQR(
    A, B, C, D, Q, R, include_output_estimates=True
)

kfpss.describe()

In [ ]:
xkp1_est = cas.DM([0.1, -0.2])
kf_xk = xkp1_est
t = cas.DM(0)
uk = cas.DM(1)

kf_xkp1 = kfpss.F(t, kf_xk, uk)
kf_y = kfpss.H(t, kf_xk, uk)
kf_xkp1, kf_y

In [ ]:
kfp = KFP(A, B, C, D, Q, R, include_output_estimates=True)

kfp.describe()

In [ ]:
xkp1_est = cas.DM([0.1, -0.2])
Pkp1 = cas.DM([[0.25, 0.0], [0.0, 0.25]])
kf_xk = cas.vertcat(xkp1_est, Pkp1.reshape((-1, 1)))
t = cas.DM(0)
uk = cas.DM(1)

kf_xkp1 = kfp.F(t, kf_xk, uk)
kf_y = kfp.H(t, kf_xk, uk)
kf_xkp1, kf_y

In [ ]:
kffss = make_steady_state_KFF_with_fixedABCDQR(
    A, B, C, D, Q, R, include_output_estimates=True
)

kffss.describe()

In [ ]:
xkp1_est = cas.DM([0.1, -0.2])
kf_xk = xkp1_est
t = cas.DM(0)
uk = cas.DM(1)

kf_xkp1 = kffss.F(t, kf_xk, uk)
kf_y = kffss.H(t, kf_xk, uk)
kf_xkp1, kf_y

In [ ]:
kff = KFF(A, B, C, D, Q, R, include_output_estimates=True)

kff.describe()

In [ ]:
xkp1_est = cas.DM([0.1, -0.2])
Pkp1 = cas.DM([[0.25, 0.0], [0.0, 0.25]])
kf_xk = cas.vertcat(xkp1_est, Pkp1.reshape((-1, 1)))
t = cas.DM(0)
uk = cas.DM(1)

kf_xkp1 = kff.F(t, kf_xk, uk)
kf_y = kff.H(t, kf_xk, uk)
kf_xkp1, kf_y

## Simulation

In [ ]:
from cas_models.continuous_time.models import SSModelCTLinearO2SISO
from cas_models.discrete_time.models import SSModelDTARXSISO, StateSpaceModelDTFromCT, StateSpaceModelDTTFSISO

# Plant
plant_ct = SSModelCTLinearO2SISO(K=1.2, T1=1.5, T2=6.0)

# Sample time
Ts = 0.25
plant_dt = StateSpaceModelDTFromCT(plant_ct, Ts)
plant_dt.describe()

In [ ]:
rng = np.random.default_rng(0)

# Simulation time
nT = 100
t = Ts * np.arange(nT+1)

# Random input signal
Wp = rng.normal(scale=0.1, size=(nT+1, 1))

# Random walk disturbance
P = np.cumsum(Wp, axis=0)

# Measurement noise
V = rng.normal(scale=0.25, size=(nT+1, 1))

assert P.shape == V.shape

In [ ]:
# Plant model
plant_model = SSModelDTARXSISO(na=2, nb=1, nk=1, dt=Ts, name='plant')
plant_model.describe()

In [ ]:
# Input disturbance model
in_dist_model = StateSpaceModelDTTFSISO([1.0], [1.0, -1.0], dt=Ts, name='in_dist')
in_dist_model.describe()

In [ ]:
# Augmented system model
model_aug = plant_model * in_dist_model
model_aug.describe()